In [1]:
# @title 1.1 🔍 Check GPU
import torch

print("🔍 GPU Check:")
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        mem_gb = props.total_memory / 1024**3
        print(f"   GPU {i}: {props.name} ({mem_gb:.1f} GB)")
else:
    print("❌ GPU not found!")

🔍 GPU Check:
   GPU 0: Tesla T4 (14.7 GB)


In [2]:
# @title 1.2 📦 Install Dependencies & Setup
import os
import sys

# ============================================
# 🚀 DEMO MODE - Set to False for full training
# ============================================
DEMO_MODE = True  # True: small subset for testing, False: full training

if DEMO_MODE:
    print("🧪 DEMO MODE ENABLED - Using small subset for testing")
    DEMO_SAMPLES = 50  # Number of samples in demo mode
    DEMO_MAX_STEPS = 20  # Max training steps in demo
    DEMO_GROUP_SIZE = 4  # Smaller group for faster training
    DEMO_MAX_TOKENS = 256  # Fewer tokens for speed
    DEMO_BATCH_SIZE = 4  # Batch size (increase if GPU memory allows)
else:
    print("🔥 FULL TRAINING MODE")
    DEMO_SAMPLES = None
    DEMO_MAX_STEPS = None
    DEMO_GROUP_SIZE = 4
    DEMO_MAX_TOKENS = 256
    DEMO_BATCH_SIZE = 4

# Install dependencies
print("\n📦 Installing RL training dependencies...")
!pip install -q transformers>=4.36.0
!pip install -q peft>=0.7.0
!pip install -q accelerate>=0.25.0
!pip install -q bitsandbytes>=0.41.0
!pip install -q trl>=0.7.0
!pip install -q datasets huggingface_hub numpy

# Additional dependencies for RAG
print("📦 Installing RAG dependencies...")
!pip install -q sentence-transformers faiss-cpu

print("\n✅ Installation Complete!")
print("   Using: transformers + peft + RAG (sentence-transformers + faiss)")
if DEMO_MODE:
    print(f"   Demo: batch_size={DEMO_BATCH_SIZE}, group_size={DEMO_GROUP_SIZE}, max_tokens={DEMO_MAX_TOKENS}")

🧪 DEMO MODE ENABLED - Using small subset for testing

📦 Installing RL training dependencies...
📦 Installing RAG dependencies...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 81.6 MB/s eta 0:00:00:00:0100:01

✅ Installation Complete!
   Using: transformers + peft + RAG (sentence-transformers + faiss)


In [4]:
# @title 1.3 🔑 Setup HuggingFace Token
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

try:
    secrets = UserSecretsClient()
    HF_TOKEN = secrets.get_secret("HF_TOKEN")
    login(token=HF_TOKEN)
    print("✅ Logged in to HuggingFace!")
except Exception as e:
    print("⚠️ Could not get HF_TOKEN from secrets.")
    HF_TOKEN = input("Enter HuggingFace Token: ")
    if HF_TOKEN:
        login(token=HF_TOKEN)
        print("✅ Logged in!")

⚠️ Could not find 'HF_TOKEN' in Colab Secrets.


---
## 2️⃣ 📊 Reward Functions

In [ ]:
# @title 2.1 🎯 Define Reward Functions (with RAG Bonus & Image-Aware Logic)
import json
import re
import numpy as np
from dataclasses import dataclass
from typing import List, Dict, Any, Optional

@dataclass
class RewardConfig:
    """Reward weights configuration"""
    iou_weight: float = 0.3
    acc_weight: float = 0.4
    format_weight: float = 0.2
    step_penalty: float = -0.1
    rag_bonus: float = 0.1  # Bonus for using RAG on hard cases
    
    # NEW: Image-aware rewards
    rag_no_image_bonus: float = 0.2  # Bonus for using RAG when no image
    image_tool_no_image_penalty: float = -0.15  # Penalty for using image tools without image

# List of image-processing tools
IMAGE_TOOLS = ['groundingdino', 'grounding_dino', 'sam', 'segment', 'detect', 'zoom', 'biomedclip']

class RewardFunction:
    """
    Composite Reward Function for TriMedAgent with RAG Bonus & Image-Aware Logic
    
    R_total = w1*R_IoU + w2*R_Acc + w3*R_Format + R_Step + R_RAG + R_Image_Aware
    
    Expected JSON Format (4 fields):
    {
        "thought": "reasoning...",
        "tool": "Vision|Knowledge",
        "action": "GroundingDINO|Medical_RAG|...",
        "action_input": {...}
    }
    
    RAG Bonus Logic:
    - If Agent uses action: "Medical_RAG" AND difficulty == "hard" → +0.1 bonus
    - If Agent uses RAG when image is None/invalid → +0.2 bonus (smart choice!)
    
    Image-Aware Logic:
    - If Agent uses image tools (GroundingDINO, SAM, etc.) when image is None → -0.15 penalty
    - Encourages Agent to recognize when image is unavailable and use RAG instead
    """
    
    def __init__(self, config: RewardConfig = None):
        self.config = config or RewardConfig()
    
    def compute_iou(self, box1: List[float], box2: List[float]) -> float:
        """Compute IoU between two boxes [x1,y1,x2,y2]"""
        x1 = max(box1[0], box2[0])
        y1 = max(box1[1], box2[1])
        x2 = min(box1[2], box2[2])
        y2 = min(box1[3], box2[3])
        
        inter = max(0, x2-x1) * max(0, y2-y1)
        area1 = (box1[2]-box1[0]) * (box1[3]-box1[1])
        area2 = (box2[2]-box2[0]) * (box2[3]-box2[1])
        union = area1 + area2 - inter
        
        return inter / union if union > 0 else 0
    
    def compute_iou_reward(
        self,
        predicted_boxes: List[List[float]],
        ground_truth_boxes: List[List[float]]
    ) -> float:
        """IoU reward - detection accuracy"""
        if not predicted_boxes or not ground_truth_boxes:
            return 0.0
        
        ious = []
        for gt_box in ground_truth_boxes:
            best_iou = max(self.compute_iou(pred, gt_box) for pred in predicted_boxes)
            ious.append(best_iou)
        
        return np.mean(ious)
    
    def compute_format_reward(self, response: str) -> float:
        """Format reward - valid JSON structure (4-field format)"""
        try:
            match = re.search(r'\{.*\}', response, re.DOTALL)
            if match:
                data = json.loads(match.group(0))
                score = 0.0
                if 'thought' in data:
                    score += 0.3
                if 'tool' in data:  # Check for 'tool' field
                    score += 0.2
                if 'action' in data:
                    score += 0.3
                if 'action_input' in data:
                    score += 0.2
                return score
        except:
            pass
        return 0.0
    
    def compute_accuracy_reward(
        self,
        predicted_answer: str,
        ground_truth_answer: str
    ) -> float:
        """Answer accuracy reward"""
        if not predicted_answer or not ground_truth_answer:
            return 0.5  # Neutral
        
        pred_lower = predicted_answer.lower()
        gt_lower = ground_truth_answer.lower()
        
        # Exact match
        if pred_lower == gt_lower:
            return 1.0
        
        # Partial match (keyword overlap)
        pred_words = set(pred_lower.split())
        gt_words = set(gt_lower.split())
        overlap = len(pred_words & gt_words) / max(len(gt_words), 1)
        
        return min(1.0, overlap)
    
    def compute_step_reward(self, num_steps: int) -> float:
        """Step efficiency reward (penalize too many steps)"""
        if num_steps <= 2:
            return 0.1
        elif num_steps <= 4:
            return 0.0
        else:
            return self.config.step_penalty * (num_steps - 4)
    
    def _extract_action(self, response: str) -> str:
        """Extract action from response"""
        try:
            match = re.search(r'\{.*\}', response, re.DOTALL)
            if match:
                data = json.loads(match.group(0))
                return data.get('action', '').lower()
        except:
            pass
        return ''
    
    def _is_rag_action(self, action: str) -> bool:
        """Check if action is RAG-related"""
        return 'medical_rag' in action or 'rag' in action or 'knowledge' in action
    
    def _is_image_tool_action(self, action: str) -> bool:
        """Check if action is image-processing tool"""
        return any(tool in action for tool in IMAGE_TOOLS)
    
    def compute_rag_bonus(self, response: str, difficulty: str, has_valid_image: bool = True) -> float:
        """
        RAG Bonus Reward (Enhanced with Image-Aware Logic)
        
        Logic:
        1. If NO valid image → Using RAG is highly rewarded (+0.2)
        2. If has image + hard case → Using RAG gives +0.1
        3. If has image + medium case → Using RAG gives +0.05
        """
        action = self._extract_action(response)
        
        if not self._is_rag_action(action):
            return 0.0
        
        # Case 1: No valid image → RAG is the smart choice!
        if not has_valid_image:
            return self.config.rag_no_image_bonus  # +0.2
        
        # Case 2: Has image, check difficulty
        if difficulty == 'hard':
            return self.config.rag_bonus  # +0.1
        elif difficulty == 'medium':
            return self.config.rag_bonus * 0.5  # +0.05
        
        return 0.0
    
    def compute_image_tool_penalty(self, response: str, has_valid_image: bool = True) -> float:
        """
        Penalty for using image tools when no valid image exists
        
        Logic:
        - If NO valid image AND agent uses image tools → PENALTY (-0.15)
        - This teaches agent to recognize when image is unavailable
        """
        if has_valid_image:
            return 0.0  # No penalty if image is valid
        
        action = self._extract_action(response)
        
        # Using image tools without image = wasteful/wrong
        if self._is_image_tool_action(action):
            return self.config.image_tool_no_image_penalty  # -0.15
        
        return 0.0
    
    def compute(
        self,
        response: str,
        predicted_boxes: List[List[float]] = None,
        ground_truth_boxes: List[List[float]] = None,
        predicted_answer: str = None,
        ground_truth_answer: str = None,
        num_steps: int = 1,
        difficulty: str = "medium",
        has_valid_image: bool = True  # NEW: image validity flag
    ) -> Dict[str, float]:
        """
        Compute total reward with RAG bonus & Image-Aware Logic
        
        Args:
            has_valid_image: False if image is None, full black, or invalid
        """
        
        r_iou = self.compute_iou_reward(
            predicted_boxes or [], ground_truth_boxes or []
        )
        r_format = self.compute_format_reward(response)
        r_acc = self.compute_accuracy_reward(
            predicted_answer or "", ground_truth_answer or ""
        )
        r_step = self.compute_step_reward(num_steps)
        
        # NEW: Image-aware rewards
        r_rag = self.compute_rag_bonus(response, difficulty, has_valid_image)
        r_image_penalty = self.compute_image_tool_penalty(response, has_valid_image)
        
        # Adjust IoU weight if no valid image (IoU is irrelevant)
        effective_iou_weight = self.config.iou_weight if has_valid_image else 0.0
        effective_acc_weight = self.config.acc_weight + (0.0 if has_valid_image else 0.1)
        
        total = (
            effective_iou_weight * r_iou +
            effective_acc_weight * r_acc +
            self.config.format_weight * r_format +
            r_step +
            r_rag +
            r_image_penalty  # Add penalty (negative value)
        )
        
        return {
            'total': total,
            'iou': r_iou,
            'format': r_format,
            'accuracy': r_acc,
            'step': r_step,
            'rag_bonus': r_rag,
            'image_penalty': r_image_penalty,  # Track penalty separately
            'has_valid_image': has_valid_image
        }

reward_fn = RewardFunction()
print("✅ Reward functions defined (with RAG Bonus & Image-Aware Logic)!")
print(f"   Weights: IoU={reward_fn.config.iou_weight}, Acc={reward_fn.config.acc_weight}, Format={reward_fn.config.format_weight}")
print(f"   RAG Bonus: +{reward_fn.config.rag_bonus} (hard) | +{reward_fn.config.rag_no_image_bonus} (no image)")
print(f"   Image Tool Penalty: {reward_fn.config.image_tool_no_image_penalty} (when no valid image)")

In [ ]:
# @title 2.2 🧪 Test Reward Function (with Image-Aware Logic)
print("="*60)
print("🧪 TESTING REWARD FUNCTION WITH IMAGE-AWARE LOGIC")
print("="*60)

# Test 1: Normal case with valid image
test_response_normal = '{"thought": "Detecting nodules", "tool": "Vision", "action": "GroundingDINO", "action_input": {"prompt": "lung nodule"}}'
print("\n📊 Test 1: Image tool WITH valid image → OK")
reward1 = reward_fn.compute(
    response=test_response_normal,
    predicted_boxes=[[100, 100, 200, 200]],
    ground_truth_boxes=[[110, 110, 210, 210]],
    predicted_answer="nodule detected",
    ground_truth_answer="lung nodule",
    num_steps=2,
    difficulty="easy",
    has_valid_image=True  # Image is valid
)
for k, v in reward1.items():
    if isinstance(v, float):
        print(f"   {k}: {v:.4f}")
    else:
        print(f"   {k}: {v}")

# Test 2: Using image tool when NO image → Should get PENALTY
test_response_image_tool = '{"thought": "I will detect abnormalities", "tool": "Vision", "action": "GroundingDINO", "action_input": {"prompt": "lung opacity"}}'
print("\n📊 Test 2: Image tool WITHOUT valid image → PENALTY!")
reward2 = reward_fn.compute(
    response=test_response_image_tool,
    predicted_boxes=[],
    ground_truth_boxes=[[100, 100, 200, 200]],
    predicted_answer="trying to detect",
    ground_truth_answer="lung opacity",
    num_steps=2,
    difficulty="medium",
    has_valid_image=False  # NO IMAGE!
)
for k, v in reward2.items():
    if isinstance(v, float):
        print(f"   {k}: {v:.4f}")
    else:
        print(f"   {k}: {v}")

# Test 3: Using RAG when NO image → Should get HIGH BONUS
test_response_rag_no_image = '{"thought": "No image provided, I will use medical knowledge", "tool": "Knowledge", "action": "Medical_RAG", "action_input": {"query": "what are symptoms of pneumonia"}}'
print("\n📊 Test 3: RAG tool WITHOUT valid image → HIGH BONUS!")
reward3 = reward_fn.compute(
    response=test_response_rag_no_image,
    predicted_boxes=[],
    ground_truth_boxes=[],
    predicted_answer="pneumonia symptoms include fever, cough",
    ground_truth_answer="fever, cough, shortness of breath",
    num_steps=1,
    difficulty="medium",
    has_valid_image=False  # NO IMAGE → RAG is smart choice!
)
for k, v in reward3.items():
    if isinstance(v, float):
        print(f"   {k}: {v:.4f}")
    else:
        print(f"   {k}: {v}")

# Test 4: RAG on hard case with valid image
test_response_rag_hard = '{"thought": "Complex case, need medical knowledge", "tool": "Knowledge", "action": "Medical_RAG", "action_input": {"query": "bilateral consolidation differential"}}'
print("\n📊 Test 4: RAG on HARD case with image → Difficulty bonus")
reward4 = reward_fn.compute(
    response=test_response_rag_hard,
    predicted_boxes=[],
    ground_truth_boxes=[[100, 100, 200, 200]],
    predicted_answer="bilateral consolidation detected",
    ground_truth_answer="bilateral pneumonia",
    num_steps=2,
    difficulty="hard",
    has_valid_image=True
)
for k, v in reward4.items():
    if isinstance(v, float):
        print(f"   {k}: {v:.4f}")
    else:
        print(f"   {k}: {v}")

print("\n" + "="*60)
print("📋 SUMMARY:")
print(f"   Test 1 (Image tool + image):     Total = {reward1['total']:.4f}")
print(f"   Test 2 (Image tool + NO image):  Total = {reward2['total']:.4f} ← PENALTY!")
print(f"   Test 3 (RAG + NO image):         Total = {reward3['total']:.4f} ← HIGH BONUS!")
print(f"   Test 4 (RAG + hard case):        Total = {reward4['total']:.4f}")
print("="*60)

---
## 3️⃣ 📊 Load RL Dataset from HuggingFace

**Simplified Data Loading:**
- Load pre-processed dataset directly from HuggingFace Hub
- Dataset: `nhn309261/medical-rl-grpo-v2` (balanced-v2 revision)
- Contains: prompts, ground_truth_boxes, ground_truth_answer, difficulty levels

**RL Training Rewards:**
- **IoU Reward**: Predicted boxes vs ground truth boxes
- **Accuracy Reward**: Answer similarity
- **Format Reward**: Valid JSON structure
- **RAG Bonus**: +0.1 for using Medical_RAG on hard cases

In [ ]:
# @title 3.1 📥 Load RL Dataset from HuggingFace
from datasets import load_dataset
from collections import Counter

print("="*60)
print("📥 LOADING RL DATASET FROM HUGGINGFACE")
print("="*60)

# Load dataset with streaming for efficiency
print("📥 Loading: nhn309261/medical-rl-grpo-v2 (balanced-v2)")

try:
    # Load streaming dataset
    rl_dataset_stream = load_dataset(
        "nhn309261/medical-rl-grpo-v2", 
        revision="balanced-v2", 
        split="train", 
        streaming=True
    )
    
    # Convert to list (with limit for demo mode)
    print("📦 Processing dataset...")
    
    rl_data_list = []
    max_samples = DEMO_SAMPLES if DEMO_MODE else None
    
    for i, item in enumerate(rl_dataset_stream):
        if max_samples and i >= max_samples:
            break
        rl_data_list.append(item)
        if (i + 1) % 100 == 0:
            print(f"   Loaded {i + 1} samples...")
    
    print(f"\n✅ Dataset loaded: {len(rl_data_list)} samples")
    
    # Show stats
    if rl_data_list:
        difficulties = Counter(item.get('difficulty', 'unknown') for item in rl_data_list)
        print(f"\n📊 Difficulty Distribution:")
        for diff, count in difficulties.most_common():
            pct = count / len(rl_data_list) * 100
            print(f"   - {diff}: {count} ({pct:.1f}%)")
        
        # Show sample
        print(f"\n📋 Sample item:")
        sample = rl_data_list[0]
        for k, v in sample.items():
            if isinstance(v, str) and len(v) > 100:
                print(f"   {k}: {v[:100]}...")
            else:
                print(f"   {k}: {v}")

except Exception as e:
    print(f"⚠️ Could not load from HuggingFace: {e}")
    print("📦 Using fallback sample data...")
    
    # Fallback sample data
    rl_data_list = [
        {
            "prompt": "Tìm các nốt mờ bất thường trong phổi.",
            "image": None,
            "ground_truth_boxes": [[100, 150, 200, 250], [300, 200, 400, 300]],
            "ground_truth_answer": "lung nodule detected in upper right and lower left regions",
            "difficulty": "easy"
        },
        {
            "prompt": "Kiểm tra dấu hiệu viêm phổi và segment vùng tổn thương.",
            "image": None,
            "ground_truth_boxes": [[200, 100, 350, 280], [180, 300, 320, 420]],
            "ground_truth_answer": "bilateral pneumonia consolidation with ground glass opacities",
            "difficulty": "hard"
        },
        {
            "prompt": "Đánh giá kích thước tim và phát hiện cardiomegaly.",
            "image": None,
            "ground_truth_boxes": [[150, 150, 400, 380]],
            "ground_truth_answer": "cardiomegaly with cardiothoracic ratio > 0.5",
            "difficulty": "medium"
        },
    ] * 20  # Expand for demo
    
    print(f"✅ Fallback data: {len(rl_data_list)} samples")

---
## 3.5️⃣ 🧠 Setup MedQuAD RAG System

**Medical RAG (Retrieval-Augmented Generation):**
- **Knowledge Base**: MedQuAD - Medical Question Answering Dataset
- **Embeddings**: sentence-transformers (all-MiniLM-L6-v2)
- **Index**: FAISS for fast similarity search
- **Tool**: `medical_rag_tool(query)` returns Top-1 relevant answer

**RAG Integration Logic:**
- **Without image**: System automatically uses RAG for theoretical knowledge
- **With image**: System orchestrates RAG when lacking domain expertise

In [ ]:
# @title 3.6 🧠 Initialize MedQuAD RAG System
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

print("="*60)
print("🧠 INITIALIZING MedQuAD RAG SYSTEM")
print("="*60)

# Load MedQuAD dataset
print("📥 Loading MedQuAD dataset...")
try:
    medquad_dataset = load_dataset("keivalya/MedQuad-MedicalQnADataset", split="train")
    print(f"   Loaded {len(medquad_dataset)} Q&A pairs")
except Exception as e:
    print(f"⚠️ Error loading MedQuAD: {e}")
    # Fallback minimal data
    medquad_dataset = None

# Initialize embedding model
print("\n📦 Loading embedding model: all-MiniLM-L6-v2")
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
print("   Embedding dimension: 384")

# Build FAISS index
print("\n🔧 Building FAISS index...")

if medquad_dataset:
    # Extract answers for indexing
    medquad_answers = [item['Answer'] for item in medquad_dataset]
    medquad_questions = [item['Question'] for item in medquad_dataset]
    
    # Limit for demo mode
    if DEMO_MODE:
        max_index = min(5000, len(medquad_answers))
        medquad_answers = medquad_answers[:max_index]
        medquad_questions = medquad_questions[:max_index]
        print(f"   DEMO MODE: Indexing {max_index} answers")
    
    # Create embeddings
    print("   Encoding answers...")
    answer_embeddings = embedding_model.encode(
        medquad_answers, 
        show_progress_bar=True,
        convert_to_numpy=True
    )
    
    # Build FAISS index
    dimension = answer_embeddings.shape[1]
    faiss_index = faiss.IndexFlatIP(dimension)  # Inner product (cosine similarity)
    
    # Normalize for cosine similarity
    faiss.normalize_L2(answer_embeddings)
    faiss_index.add(answer_embeddings)
    
    print(f"   ✅ FAISS index built: {faiss_index.ntotal} vectors")
else:
    faiss_index = None
    medquad_answers = []
    medquad_questions = []
    print("   ⚠️ Using fallback (no index)")

# Define RAG tool function
def medical_rag_tool(query: str, top_k: int = 1) -> str:
    """
    Medical RAG Tool - Retrieve relevant medical knowledge
    
    Args:
        query: Medical question or search query
        top_k: Number of results to return (default: 1)
    
    Returns:
        Most relevant medical answer from MedQuAD
    """
    if faiss_index is None or faiss_index.ntotal == 0:
        return "No medical knowledge base available."
    
    # Encode query
    query_embedding = embedding_model.encode([query], convert_to_numpy=True)
    faiss.normalize_L2(query_embedding)
    
    # Search
    scores, indices = faiss_index.search(query_embedding, top_k)
    
    results = []
    for i, (score, idx) in enumerate(zip(scores[0], indices[0])):
        if idx < len(medquad_answers):
            answer = medquad_answers[idx]
            question = medquad_questions[idx] if idx < len(medquad_questions) else ""
            results.append({
                'question': question,
                'answer': answer,
                'score': float(score)
            })
    
    if results:
        # Return top-1 answer
        return results[0]['answer']
    
    return "No relevant medical information found."

print("\n✅ MedQuAD RAG System Ready!")
print(f"   Index size: {faiss_index.ntotal if faiss_index else 0} documents")
print(f"   Tool: medical_rag_tool(query) → Top-1 answer")

In [ ]:
# @title 3.7 🧪 Test RAG Tool
print("🧪 Testing Medical RAG Tool...")
print("="*50)

test_queries = [
    "What are the symptoms of pneumonia?",
    "How is cardiomegaly diagnosed?",
    "What causes pleural effusion?",
]

for query in test_queries:
    print(f"\n📝 Query: {query}")
    answer = medical_rag_tool(query)
    # Truncate long answers for display
    if len(answer) > 300:
        answer = answer[:300] + "..."
    print(f"📚 Answer: {answer}")

print("\n" + "="*50)
print("✅ RAG Tool working correctly!")

---
## 4️⃣ 🚀 GRPO Training Configuration

In [ ]:
# @title 4.1 ⚙️ GRPO Config (with Demo Mode Support)
from dataclasses import dataclass, field
from typing import List

@dataclass
class GRPOConfig:
    # Model - LLaVA-Med for medical imaging
    base_model: str = "chaoyinshe/llava-med-v1.5-mistral-7b-hf"
    sft_adapter: str = "nhn309261/trimedagent-sft"  # From Stage 1
    
    # GRPO hyperparameters - adjusted for demo mode
    group_size: int = DEMO_GROUP_SIZE if DEMO_MODE else 4
    temperature: float = 0.8  # Slightly higher for more diverse outputs
    beta: float = 0.1  # KL penalty
    
    # Training - Use GPU efficiently with batching
    max_steps: int = DEMO_MAX_STEPS if DEMO_MODE else 200
    batch_size: int = DEMO_BATCH_SIZE if DEMO_MODE else 4  # 🔥 INCREASED BATCH SIZE
    gradient_accumulation_steps: int = 2  # Effective batch = batch_size * grad_accum
    learning_rate: float = 1e-5
    max_new_tokens: int = DEMO_MAX_TOKENS if DEMO_MODE else 256
    
    # RAG settings
    use_rag: bool = True  # Enable RAG integration
    rag_threshold_difficulty: str = "medium"  # Use RAG for medium+ difficulty
    
    # Curriculum
    use_curriculum: bool = True
    curriculum_levels: List[str] = field(default_factory=lambda: ['easy', 'medium', 'hard'])
    
    # Output
    output_dir: str = "checkpoints/rl_adapter"
    hub_model_id: str = "nhn309261/trimedagent-grpo"

config = GRPOConfig()
print("✅ GRPO Config ready!")
print(f"   Base Model: {config.base_model}")
print(f"   SFT Adapter: {config.sft_adapter}")
print(f"   🔥 Batch Size: {config.batch_size}" + (" (DEMO)" if DEMO_MODE else ""))
print(f"   Group Size: {config.group_size}" + (" (DEMO)" if DEMO_MODE else ""))
print(f"   Effective Batch: {config.batch_size * config.gradient_accumulation_steps}")
print(f"   Max New Tokens: {config.max_new_tokens}" + (" (DEMO)" if DEMO_MODE else ""))
print(f"   Max Steps: {config.max_steps}" + (" (DEMO)" if DEMO_MODE else ""))
print(f"   RAG Enabled: {config.use_rag}")

In [ ]:
# @title 4.2 📚 Load Model with SFT Adapter
from transformers import AutoTokenizer, LlavaForConditionalGeneration, BitsAndBytesConfig
from peft import PeftModel
import torch

print(f"🚀 Loading base model: {config.sft_adapter}")
print("   (LLaVA-Med - specialized for medical imaging)")

# Quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True
)

# Tokenizer
tokenizer = AutoTokenizer.from_pretrained(config.sft_adapter, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

# Base model - LLaVA-Med
sft_adapter = LlavaForConditionalGeneration.from_pretrained(
    config.sft_adapter,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    torch_dtype=torch.bfloat16
)

print(f"📥 Loading SFT adapter: {config.sft_adapter}")
model = PeftModel.from_pretrained(
    sft_adapter,
    config.sft_adapter,
    is_trainable=True
)

# Enable gradient checkpointing
model.gradient_checkpointing_enable()

print("✅ LLaVA-Med loaded with SFT adapter!")

In [ ]:
# @title 4.3 📊 Create RLDataset with JSON-Instruction Prompt
from torch.utils.data import Dataset, DataLoader
import json

# System prompt that instructs model to output valid JSON
SYSTEM_PROMPT = """You are TriMedAgent, a medical visual analysis assistant.
You MUST respond with a valid JSON object in this EXACT format:
{
    "thought": "Your reasoning about the medical image/question",
    "tool": "Vision or Knowledge",
    "action": "GroundingDINO or Medical_RAG or SAM or BioMedCLIP",
    "action_input": {"query": "your query here"} or {"prompt": "detection prompt"}
}

Available tools:
- Vision tools (require image): GroundingDINO, SAM, BioMedCLIP
- Knowledge tools (no image needed): Medical_RAG

If no image is provided, use Medical_RAG to retrieve knowledge."""

# Few-shot example to guide the model
FEW_SHOT_EXAMPLE = """Example 1 (with image):
Question: Detect lung nodules in this chest X-ray.
Response: {"thought": "I need to detect lung nodules in the chest X-ray image", "tool": "Vision", "action": "GroundingDINO", "action_input": {"prompt": "lung nodule"}}

Example 2 (without image):
Question: What are the symptoms of pneumonia?
Response: {"thought": "No image provided, I will use medical knowledge base", "tool": "Knowledge", "action": "Medical_RAG", "action_input": {"query": "symptoms of pneumonia"}}"""

class RLDataset(Dataset):
    """
    RL Dataset for GRPO Training with JSON-Instruction Prompt
    
    Supports both:
    - Pre-loaded list data (from HuggingFace streaming)
    - Local JSONL files (fallback)
    """
    
    def __init__(self, data, tokenizer, max_length=2048, difficulty_filter=None):
        """
        Args:
            data: Either a list of dicts or path to JSONL file
            tokenizer: HuggingFace tokenizer
            max_length: Max sequence length
            difficulty_filter: Filter by difficulty ('easy', 'medium', 'hard')
        """
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.data = []
        
        # Handle different input types
        if isinstance(data, list):
            # Already loaded data (from HuggingFace)
            self.data = data
        elif isinstance(data, str):
            # Path to JSONL file
            with open(data, 'r', encoding='utf-8') as f:
                for line in f:
                    if line.strip():
                        self.data.append(json.loads(line))
        else:
            raise ValueError(f"Unsupported data type: {type(data)}")
        
        # Filter by difficulty if specified
        if difficulty_filter:
            self.data = [
                item for item in self.data 
                if item.get('difficulty') == difficulty_filter
            ]
        
        print(f"📊 RLDataset: {len(self.data)} examples" + 
              (f" (difficulty={difficulty_filter})" if difficulty_filter else ""))
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        item = self.data[idx]
        
        # Handle both 'prompt' and 'question' keys
        prompt_text = item.get('prompt') or item.get('question', '')
        
        # Determine if image is available
        has_image = item.get('image') is not None or item.get('has_image', False)
        image_context = "An image is provided." if has_image else "No image is provided."
        
        # Format prompt with system instruction + few-shot + question
        full_prompt = f"""<s>[INST] {SYSTEM_PROMPT}

{FEW_SHOT_EXAMPLE}

{image_context}
Question: {prompt_text}
Response: [/INST]"""
        
        encodings = self.tokenizer(
            full_prompt,
            truncation=True,
            max_length=self.max_length,
            return_tensors='pt'
        )
        
        # Handle different key formats
        gt_boxes = item.get('ground_truth_boxes') or item.get('boxes', [])
        gt_answer = item.get('ground_truth_answer') or item.get('answer', '')
        difficulty = item.get('difficulty', 'medium')
        
        return {
            'input_ids': encodings['input_ids'].squeeze(),
            'attention_mask': encodings['attention_mask'].squeeze(),
            'prompt': prompt_text,
            'ground_truth_boxes': gt_boxes,
            'ground_truth_answer': gt_answer,
            'difficulty': difficulty,
            'has_image': has_image  # For RAG decision
        }

# Create dataset from loaded data
print("\n📦 Creating RLDataset with JSON-Instruction Prompt...")
full_dataset = RLDataset(rl_data_list, tokenizer)

# Show dataset info
print(f"\n📊 Dataset Summary:")
print(f"   Total samples: {len(full_dataset)}")

# Difficulty distribution
from collections import Counter
difficulties = Counter(item.get('difficulty', 'unknown') for item in rl_data_list)
for diff, count in difficulties.most_common():
    print(f"   - {diff}: {count} ({count/len(rl_data_list)*100:.1f}%)")

# Show sample prompt
print(f"\n📋 Sample formatted prompt (first 500 chars):")
sample = full_dataset[0]
sample_text = tokenizer.decode(sample['input_ids'][:200])
print(f"   {sample_text[:500]}...")

---
## 5️⃣ 🔥 GRPO Training Loop

In [ ]:
# @title 5.1 🎯 GRPO Training Functions (Optimized with RAG Integration)
from transformers import GenerationConfig
import torch.nn.functional as F
from tqdm import tqdm
from collections import defaultdict

def generate_responses(model, tokenizer, prompt_ids, attention_mask, num_responses, max_new_tokens=150, temperature=0.8):
    """Generate multiple responses for GRPO (without gradients for efficiency)"""
    model.eval()
    responses = []
    response_ids_list = []
    
    gen_config = GenerationConfig(
        max_new_tokens=max_new_tokens,
        temperature=temperature,
        do_sample=True,
        top_p=0.9,
        top_k=50,  # Add top_k for better quality
        repetition_penalty=1.1,  # Avoid repetition
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )
    
    with torch.no_grad():
        for _ in range(num_responses):
            outputs = model.generate(
                input_ids=prompt_ids,
                attention_mask=attention_mask,
                generation_config=gen_config,
                return_dict_in_generate=True
            )
            
            response_ids = outputs.sequences[0, prompt_ids.shape[1]:]
            response_text = tokenizer.decode(response_ids, skip_special_tokens=True)
            responses.append(response_text)
            response_ids_list.append(response_ids)
    
    return responses, response_ids_list

def compute_log_probs_with_grad(model, tokenizer, prompt_ids, attention_mask, response_ids_list):
    """
    Recompute log probabilities WITH gradients for policy gradient
    
    This is necessary because generate() runs with torch.no_grad()
    We need to do a forward pass to get gradients for backprop
    """
    model.train()
    log_probs_list = []
    
    for response_ids in response_ids_list:
        if len(response_ids) == 0:
            log_probs_list.append(torch.zeros(1, device=model.device, requires_grad=True))
            continue
        
        # Concatenate prompt + response for forward pass
        full_ids = torch.cat([prompt_ids[0], response_ids], dim=0).unsqueeze(0)
        full_mask = torch.ones_like(full_ids)
        
        # Forward pass with gradients
        outputs = model(
            input_ids=full_ids,
            attention_mask=full_mask,
            return_dict=True
        )
        
        # Get logits for response tokens only
        logits = outputs.logits[0, prompt_ids.shape[1]-1:-1, :]  # Shift by 1 for next-token prediction
        
        # Compute log probs for actual generated tokens
        log_probs = F.log_softmax(logits, dim=-1)
        
        # Gather log probs of generated tokens
        response_log_probs = []
        for i, token_id in enumerate(response_ids):
            if i < log_probs.shape[0]:
                response_log_probs.append(log_probs[i, token_id])
        
        if response_log_probs:
            log_probs_list.append(torch.stack(response_log_probs))
        else:
            log_probs_list.append(torch.zeros(1, device=model.device, requires_grad=True))
    
    return log_probs_list

def extract_json_from_response(response):
    """Extract and parse JSON from response with multiple strategies"""
    # Strategy 1: Find JSON block
    try:
        match = re.search(r'\{[^{}]*\}', response, re.DOTALL)
        if match:
            return json.loads(match.group(0))
    except:
        pass
    
    # Strategy 2: Try to parse the whole response as JSON
    try:
        # Clean up response
        clean = response.strip()
        if clean.startswith('{') and clean.endswith('}'):
            return json.loads(clean)
    except:
        pass
    
    # Strategy 3: Extract key-value pairs manually
    result = {}
    for key in ['thought', 'tool', 'action', 'action_input']:
        match = re.search(rf'"{key}"\s*:\s*"([^"]*)"', response)
        if match:
            result[key] = match.group(1)
    
    return result if result else None

def extract_boxes_from_response(response):
    """Extract boxes from JSON response"""
    data = extract_json_from_response(response)
    if data:
        action_input = data.get('action_input', {})
        if isinstance(action_input, dict) and 'boxes' in action_input:
            return action_input['boxes']
    return []

def extract_action_from_response(response):
    """Extract action type from JSON response"""
    data = extract_json_from_response(response)
    if data:
        return data.get('action', '').lower()
    return ''

def extract_rag_query_from_response(response):
    """Extract RAG query from response if action is Medical_RAG"""
    data = extract_json_from_response(response)
    if data:
        action = data.get('action', '').lower()
        if 'rag' in action or 'medical_rag' in action or 'knowledge' in action:
            action_input = data.get('action_input', {})
            if isinstance(action_input, dict):
                return action_input.get('query', '') or action_input.get('question', '')
    return None

def count_steps(response):
    """Count action steps in response"""
    return max(1, len(re.findall(r'"action"\s*:', response)))

def simulate_rag_tool_call(response, prompt):
    """
    Simulate RAG tool call if agent requests it
    
    Returns:
        Tuple (rag_result, augmented_prompt) if RAG was called
        None if no RAG action in response
    """
    rag_query = extract_rag_query_from_response(response)
    
    if rag_query:
        # Call RAG tool
        rag_result = medical_rag_tool(rag_query)
        
        # Create augmented prompt with RAG knowledge
        augmented_prompt = f"""Based on the medical knowledge:
{rag_result}

Original question: {prompt}
Please provide your analysis."""
        
        return rag_result, augmented_prompt
    
    return None, None

print("✅ GRPO functions defined (Optimized with RAG integration)!")
print("   - generate_responses: Generate responses (no grad)")
print("   - compute_log_probs_with_grad: Recompute log probs WITH gradients")
print("   - extract_json_from_response: Multi-strategy JSON extraction")
print("   - simulate_rag_tool_call: Execute RAG when agent requests it")

In [ ]:
# @title 5.2 🚂 Run GRPO Training (Batched + GPU Optimized)
import torch.optim as optim

# Optimizer
optimizer = optim.AdamW(model.parameters(), lr=config.learning_rate, weight_decay=0.01)

# Metrics
metrics = defaultdict(list)

# Debug: Track sample responses
debug_responses = []

def grpo_step_batched(batch, step_num=0):
    """
    GRPO training step with BATCHED processing for GPU efficiency
    
    Processes multiple samples in parallel on GPU
    """
    batch_size = len(batch['prompt']) if isinstance(batch['prompt'], list) else 1
    
    # Collect all results
    all_rewards = []
    all_details = []
    all_log_probs = []
    all_rag_used = []
    
    for b_idx in range(batch_size):
        # Extract single sample from batch
        if batch_size > 1:
            input_ids = batch['input_ids'][b_idx]
            attn_mask = batch['attention_mask'][b_idx]
            gt_boxes = batch['ground_truth_boxes'][b_idx]
            gt_answer = batch['ground_truth_answer'][b_idx]
            difficulty = batch['difficulty'][b_idx]
            original_prompt = batch['prompt'][b_idx]
            has_image = batch.get('has_image', [False])[b_idx]
        else:
            input_ids = batch['input_ids']
            attn_mask = batch['attention_mask']
            gt_boxes = batch['ground_truth_boxes']
            gt_answer = batch['ground_truth_answer']
            difficulty = batch['difficulty']
            original_prompt = batch['prompt']
            has_image = batch.get('has_image', False)
        
        # Handle tensor dimensions
        if input_ids.dim() == 1:
            input_ids = input_ids.unsqueeze(0)
        if attn_mask.dim() == 1:
            attn_mask = attn_mask.unsqueeze(0)
        
        prompt_ids = input_ids.to(model.device)
        attention_mask = attn_mask.to(model.device)
        
        # Handle data types
        if isinstance(gt_boxes, str):
            try:
                gt_boxes = json.loads(gt_boxes)
            except:
                gt_boxes = []
        if isinstance(gt_boxes, (list, tuple)) and len(gt_boxes) == 1 and isinstance(gt_boxes[0], list):
            gt_boxes = gt_boxes[0] if isinstance(gt_boxes[0][0], (int, float)) else gt_boxes
        
        if not isinstance(gt_answer, str):
            gt_answer = str(gt_answer) if gt_answer else ''
        if not isinstance(difficulty, str):
            difficulty = str(difficulty) if difficulty else 'medium'
        if isinstance(has_image, torch.Tensor):
            has_image = has_image.item()
        
        # Generate responses for this sample
        responses, response_ids_list = generate_responses(
            model, tokenizer, prompt_ids, attention_mask,
            config.group_size, config.max_new_tokens, config.temperature
        )
        
        # Debug: Save first responses
        if step_num < 2 and b_idx == 0:
            debug_responses.append({
                'step': step_num,
                'prompt': original_prompt[:100] if isinstance(original_prompt, str) else str(original_prompt)[:100],
                'responses': [r[:200] for r in responses]
            })
        
        # Compute log probs with gradients
        log_probs_list = compute_log_probs_with_grad(
            model, tokenizer, prompt_ids, attention_mask, response_ids_list
        )
        
        # Process RAG and compute rewards
        for resp_idx, (resp, log_probs) in enumerate(zip(responses, log_probs_list)):
            rag_used = False
            final_resp = resp
            
            # Check for RAG
            if config.use_rag:
                rag_result, _ = simulate_rag_tool_call(resp, original_prompt)
                if rag_result:
                    rag_used = True
                    final_resp = f"{resp}\n[RAG]: {rag_result[:150]}..."
            
            # Compute reward
            pred_boxes = extract_boxes_from_response(final_resp)
            num_steps = count_steps(final_resp) + (1 if rag_used else 0)
            
            reward = reward_fn.compute(
                response=final_resp,
                predicted_boxes=pred_boxes,
                ground_truth_boxes=gt_boxes if isinstance(gt_boxes, list) else [],
                predicted_answer=final_resp[:200],
                ground_truth_answer=gt_answer,
                num_steps=num_steps,
                difficulty=difficulty,
                has_valid_image=bool(has_image)
            )
            
            all_rewards.append(reward['total'])
            all_details.append(reward)
            all_log_probs.append(log_probs)
            all_rag_used.append(rag_used)
    
    # Normalize rewards across entire batch
    rewards = np.array(all_rewards)
    rewards_normalized = (rewards - rewards.mean()) / (rewards.std() + 1e-8)
    
    # Compute policy gradient loss
    model.train()
    total_loss = 0.0
    valid_count = 0
    
    for log_probs, reward in zip(all_log_probs, rewards_normalized):
        if isinstance(log_probs, torch.Tensor) and len(log_probs) > 0:
            loss = -reward * log_probs.sum()
            total_loss += loss
            valid_count += 1
    
    if valid_count > 0:
        total_loss = total_loss / valid_count
    
    # Backward pass
    optimizer.zero_grad()
    if isinstance(total_loss, torch.Tensor):
        total_loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        loss_val = total_loss.item()
    else:
        loss_val = float(total_loss)
    
    return {
        'loss': loss_val,
        'mean_reward': float(rewards.mean()),
        'max_reward': float(rewards.max()),
        'format_reward': np.mean([d['format'] for d in all_details]),
        'rag_bonus': np.mean([d['rag_bonus'] for d in all_details]),
        'rag_usage': sum(all_rag_used) / len(all_rag_used) if all_rag_used else 0,
        'batch_size': batch_size,
    }

# Training loop
print("🚂 Starting GRPO Training (Batched + GPU Optimized)...")
print(f"   Max steps: {config.max_steps}" + (" (DEMO MODE)" if DEMO_MODE else ""))
print(f"   🔥 Batch size: {config.batch_size}")
print(f"   Group size: {config.group_size}")
print(f"   Max tokens: {config.max_new_tokens}")
print(f"   RAG enabled: {config.use_rag}")
print("="*50)

# Check GPU memory
if torch.cuda.is_available():
    free_mem = torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated(0)
    print(f"   GPU Memory Free: {free_mem / 1024**3:.1f} GB")
print("="*50)

# Custom collate for batching
def custom_collate(batch):
    """Handle batching with mixed tensor/non-tensor data"""
    result = {}
    for key in batch[0].keys():
        values = [item[key] for item in batch]
        if isinstance(values[0], torch.Tensor):
            # Pad tensors to same length
            max_len = max(v.shape[0] for v in values)
            padded = []
            for v in values:
                if v.shape[0] < max_len:
                    pad_size = max_len - v.shape[0]
                    v = F.pad(v, (0, pad_size), value=tokenizer.pad_token_id)
                padded.append(v)
            result[key] = torch.stack(padded)
        else:
            result[key] = values
    return result

dataloader = DataLoader(
    full_dataset, 
    batch_size=config.batch_size,  # 🔥 BATCHED!
    shuffle=True, 
    collate_fn=custom_collate,
    drop_last=True  # Ensure consistent batch size
)
global_step = 0
grad_accum_step = 0

pbar = tqdm(total=config.max_steps, desc="GRPO Training")

while global_step < config.max_steps:
    for batch in dataloader:
        try:
            step_metrics = grpo_step_batched(batch, global_step)
            
            for k, v in step_metrics.items():
                if k != 'batch_size':
                    metrics[k].append(v)
            
            global_step += 1
            pbar.update(1)
            
            if global_step % 5 == 0:
                avg_loss = np.mean(metrics['loss'][-5:])
                avg_reward = np.mean(metrics['mean_reward'][-5:])
                avg_format = np.mean(metrics['format_reward'][-5:]) * 100
                avg_rag = np.mean(metrics['rag_usage'][-5:]) * 100
                pbar.set_postfix({
                    'loss': f'{avg_loss:.3f}', 
                    'reward': f'{avg_reward:.3f}',
                    'fmt%': f'{avg_format:.0f}',
                    'RAG%': f'{avg_rag:.0f}'
                })
            
            if global_step >= config.max_steps:
                break
                
        except torch.cuda.OutOfMemoryError:
            print(f"\n⚠️ GPU OOM at step {global_step}! Reduce batch_size or max_tokens.")
            torch.cuda.empty_cache()
            continue
        except Exception as e:
            print(f"\n⚠️ Error at step {global_step}: {e}")
            continue

pbar.close()

print("\n✅ GRPO Training Complete!")
print(f"   Total steps: {global_step}")
print(f"   Final avg reward: {np.mean(metrics['mean_reward'][-10:]):.4f}")
print(f"   Final format accuracy: {np.mean(metrics['format_reward'][-10:])*100:.1f}%")
print(f"   Avg RAG usage: {np.mean(metrics['rag_usage'])*100:.1f}%")

# Show debug responses
if debug_responses:
    print("\n" + "="*60)
    print("🔍 DEBUG: Sample Generated Responses")
    print("="*60)
    for debug in debug_responses[:2]:
        print(f"\n📝 Step {debug['step']}: {debug['prompt']}...")
        for i, resp in enumerate(debug['responses'][:2]):
            print(f"   Response {i+1}: {resp}...")

In [ ]:
# @title 5.2.1 📋 Print Training Metrics Summary
import pandas as pd

print("="*70)
print("📋 GRPO TRAINING METRICS - DETAILED VIEW")
print("="*70)

if metrics['loss']:
    # Create DataFrame for easy viewing
    df_metrics = pd.DataFrame({
        'Step': range(1, len(metrics['loss']) + 1),
        'Loss': metrics['loss'],
        'Mean Reward': metrics['mean_reward'],
        'Max Reward': metrics['max_reward'],
        'Format': metrics['format_reward'],
        'RAG Bonus': metrics['rag_bonus'],
        'RAG Usage %': [x * 100 for x in metrics['rag_usage']]
    })
    
    # Display all rows for small datasets
    if len(df_metrics) <= 30:
        pd.set_option('display.max_rows', None)
        print("\n📊 All Training Steps:")
        print(df_metrics.to_string(index=False))
    else:
        print("\n📊 First 10 Steps:")
        print(df_metrics.head(10).to_string(index=False))
        print("\n...")
        print("\n📊 Last 10 Steps:")
        print(df_metrics.tail(10).to_string(index=False))
    
    # Summary statistics
    print("\n" + "="*70)
    print("📈 SUMMARY STATISTICS")
    print("="*70)
    print(f"{'Metric':<20} {'Min':>10} {'Max':>10} {'Mean':>10} {'Std':>10}")
    print("-"*60)
    for col in ['Loss', 'Mean Reward', 'Max Reward', 'Format', 'RAG Bonus', 'RAG Usage %']:
        data = df_metrics[col]
        print(f"{col:<20} {data.min():>10.4f} {data.max():>10.4f} {data.mean():>10.4f} {data.std():>10.4f}")
    
    print("\n" + "="*70)
    print("🎯 FINAL RESULTS (Last 5 steps average)")
    print("="*70)
    last_n = min(5, len(metrics['loss']))
    print(f"   Loss:        {np.mean(metrics['loss'][-last_n:]):.4f}")
    print(f"   Mean Reward: {np.mean(metrics['mean_reward'][-last_n:]):.4f}")
    print(f"   Max Reward:  {np.mean(metrics['max_reward'][-last_n:]):.4f}")
    print(f"   Format:      {np.mean(metrics['format_reward'][-last_n:])*100:.1f}%")
    print(f"   RAG Usage:   {np.mean(metrics['rag_usage'][-last_n:])*100:.1f}%")
    print("="*70)
else:
    print("⚠️ No metrics recorded yet. Run training first!")

In [ ]:
# @title 5.3 📊 Plot GRPO Training Metrics (with RAG)
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

# Create output directory for plots
plots_dir = Path("outputs/training_plots")
plots_dir.mkdir(parents=True, exist_ok=True)

# Helper function to plot and save
def plot_metric(ax, data, color, title, ylabel, xlabel='Training Steps', 
                window=10, ylim=None, show_legend=True, alpha=0.5, 
                label='', secondary_data=None, secondary_color=None, secondary_label=None):
    """Plot metric with optional smoothing and save individual figure"""
    ax.plot(data, alpha=alpha, color=color, label=label if label else None)
    
    # Plot secondary data if provided
    if secondary_data is not None:
        ax.plot(secondary_data, alpha=alpha, color=secondary_color, label=secondary_label)
    
    # Add smoothed line
    smoothed = None
    if len(data) >= window and window > 1:
        smoothed = np.convolve(data, np.ones(window)/window, mode='valid')
        ax.plot(range(window-1, len(data)), smoothed, color=color, linewidth=2, 
                label='Smoothed' if label else None)
    
    ax.set_xlabel(xlabel, fontsize=11)
    ax.set_ylabel(ylabel, fontsize=11)
    ax.set_title(title, fontsize=13, fontweight='bold')
    if ylim:
        ax.set_ylim(ylim)
    if show_legend and (label or secondary_label):
        ax.legend()
    ax.grid(True, alpha=0.3)
    
    return smoothed

def save_individual_plot(data, color, title, ylabel, filename, window=10, ylim=None,
                         secondary_data=None, secondary_color=None, secondary_label=None,
                         is_histogram=False, mean_line=None):
    """Save individual plot to file"""
    fig, ax = plt.subplots(figsize=(8, 6))
    
    if is_histogram:
        ax.hist(data, bins=20, alpha=0.7, color=color, edgecolor='white')
        if mean_line is not None:
            ax.axvline(x=mean_line, color='red', linestyle='--', linewidth=2, 
                      label=f'Mean: {mean_line:.3f}')
            ax.legend()
    else:
        ax.plot(data, alpha=0.5, color=color)
        if secondary_data is not None:
            ax.plot(secondary_data, alpha=0.5, color=secondary_color, label=secondary_label)
        if len(data) >= window and window > 1:
            smoothed = np.convolve(data, np.ones(window)/window, mode='valid')
            ax.plot(range(window-1, len(data)), smoothed, color=color, linewidth=2)
    
    ax.set_xlabel('Training Steps' if not is_histogram else 'Reward Value', fontsize=11)
    ax.set_ylabel(ylabel, fontsize=11)
    ax.set_title(title, fontsize=13, fontweight='bold')
    if ylim:
        ax.set_ylim(ylim)
    ax.grid(True, alpha=0.3)
    
    fig.savefig(plots_dir / filename, dpi=150, bbox_inches='tight')
    plt.close(fig)

# Setup
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
window = min(10, len(metrics['loss']) // 2) if len(metrics['loss']) > 1 else 1

# Plot configurations
plot_configs = [
    # (ax, data, color, title, ylabel, filename, ylim, extra_kwargs)
    (axes[0,0], metrics['loss'], 'blue', '📉 GRPO Policy Loss', 'Policy Loss', '01_policy_loss.png', None),
    (axes[0,2], metrics['format_reward'], 'purple', '✅ JSON Format Correctness', 'Format Reward', '03_format_reward.png', [0, 1.1]),
    (axes[1,1], metrics['rag_bonus'], 'red', '🎁 RAG Bonus Reward', 'RAG Bonus', '05_rag_bonus.png', None),
]

# Plot 1, 3, 5: Simple line plots
for ax, data, color, title, ylabel, filename, ylim in plot_configs:
    plot_metric(ax, data, color, title, ylabel, window=window, ylim=ylim)
    save_individual_plot(data, color, title, ylabel, filename, window=window, ylim=ylim)

# Plot 2: Reward Curves (dual lines)
ax2 = axes[0, 1]
ax2.plot(metrics['mean_reward'], alpha=0.5, color='green', label='Mean Reward')
ax2.plot(metrics['max_reward'], alpha=0.5, color='orange', label='Max Reward')
if len(metrics['mean_reward']) >= window and window > 1:
    smoothed_mean = np.convolve(metrics['mean_reward'], np.ones(window)/window, mode='valid')
    ax2.plot(range(window-1, len(metrics['mean_reward'])), smoothed_mean, 'g-', linewidth=2, label='Smoothed Mean')
ax2.set_xlabel('Training Steps', fontsize=11)
ax2.set_ylabel('Reward', fontsize=11)
ax2.set_title('📈 Reward Progress', fontsize=13, fontweight='bold')
ax2.legend()
ax2.grid(True, alpha=0.3)
save_individual_plot(metrics['mean_reward'], 'green', '📈 Reward Progress', 'Reward', 
                     '02_reward_curves.png', window=window,
                     secondary_data=metrics['max_reward'], secondary_color='orange', secondary_label='Max Reward')

# Plot 4: RAG Usage Rate
rag_usage_pct = [x * 100 for x in metrics['rag_usage']]
plot_metric(axes[1,0], rag_usage_pct, 'teal', '🧠 RAG Tool Usage Rate', 'RAG Usage (%)', 
            window=window, ylim=[0, 100], show_legend=False)
save_individual_plot(rag_usage_pct, 'teal', '🧠 RAG Tool Usage Rate', 'RAG Usage (%)', 
                     '04_rag_usage.png', window=window, ylim=[0, 100])

# Plot 6: Reward Distribution (histogram)
ax6 = axes[1, 2]
ax6.hist(metrics['mean_reward'], bins=20, alpha=0.7, color='teal', edgecolor='white')
mean_val = np.mean(metrics['mean_reward']) if metrics['mean_reward'] else 0
if metrics['mean_reward']:
    ax6.axvline(x=mean_val, color='red', linestyle='--', linewidth=2, label=f'Mean: {mean_val:.3f}')
ax6.set_xlabel('Reward Value', fontsize=11)
ax6.set_ylabel('Frequency', fontsize=11)
ax6.set_title('📊 Reward Distribution', fontsize=13, fontweight='bold')
ax6.legend()
ax6.grid(True, alpha=0.3)
save_individual_plot(metrics['mean_reward'], 'teal', '📊 Reward Distribution', 'Frequency',
                     '06_reward_distribution.png', is_histogram=True, mean_line=mean_val)

# Save combined figure
plt.tight_layout()
fig.savefig(plots_dir / 'grpo_training_metrics_combined.png', dpi=150, bbox_inches='tight')
plt.show()

# Print Summary
print("\n" + "="*60)
print("📊 GRPO TRAINING SUMMARY (with RAG Integration)")
print("="*60)
print(f"   Total Steps: {len(metrics['loss'])}")
print(f"   Final Loss: {np.mean(metrics['loss'][-10:]) if metrics['loss'] else 0:.4f}")
print(f"   Final Mean Reward: {np.mean(metrics['mean_reward'][-10:]) if metrics['mean_reward'] else 0:.4f}")
print(f"   Best Reward: {max(metrics['max_reward']) if metrics['max_reward'] else 0:.4f}")
print(f"   Format Accuracy: {np.mean(metrics['format_reward'][-10:])*100 if metrics['format_reward'] else 0:.1f}%")
print(f"   Avg RAG Usage: {np.mean(metrics['rag_usage'])*100 if metrics['rag_usage'] else 0:.1f}%")
print(f"   Avg RAG Bonus: {np.mean(metrics['rag_bonus']) if metrics['rag_bonus'] else 0:.4f}")
print("="*60)
print(f"\n📁 Plots saved to: {plots_dir.absolute()}")
print("   📈 01-06 individual plots + combined figure")
print("\n✅ All training plots saved!")

---
## 6️⃣ 💾 Save & Push to HuggingFace

In [ ]:
# @title 6.1 💾 Save Adapter Locally
from pathlib import Path
import json

output_path = Path(config.output_dir) / "final"
output_path.mkdir(parents=True, exist_ok=True)

# Save model
model.save_pretrained(str(output_path))
tokenizer.save_pretrained(str(output_path))

# Save metrics
with open(output_path / "training_metrics.json", 'w') as f:
    json.dump({k: list(v) for k, v in metrics.items()}, f)

# Save config
config_dict = {
    'base_model': config.base_model,
    'sft_adapter': config.sft_adapter,
    'group_size': config.group_size,
    'max_steps': config.max_steps,
    'learning_rate': config.learning_rate,
}
with open(output_path / "grpo_config.json", 'w') as f:
    json.dump(config_dict, f, indent=2)

print(f"✅ Adapter saved to: {output_path}")

In [ ]:
# @title 6.2 📤 Push to HuggingFace Hub
REPO_ID = config.hub_model_id  # e.g., "nhn309261/trimedagent-grpo-v1"

print(f"📤 Pushing to HuggingFace: {REPO_ID}")

try:
    model.push_to_hub(REPO_ID, use_auth_token=True)
    tokenizer.push_to_hub(REPO_ID, use_auth_token=True)
    
    print(f"\n✅ Successfully pushed to: https://huggingface.co/{REPO_ID}")
    
except Exception as e:
    print(f"❌ Push failed: {e}")
    print(f"\nTry manual upload:")
    print(f"   huggingface-cli upload {REPO_ID} {output_path}")

In [ ]:
# @title 6.3 📋 Create Model Card (with RAG)
model_card = f"""---
license: apache-2.0
tags:
  - medical
  - vision
  - llava
  - lora
  - grpo
  - reinforcement-learning
  - trimedagent
  - rag
base_model: {config.base_model}
---

# TriMedAgent GRPO Adapter (with Medical RAG)

LoRA adapter fine-tuned with **Group Relative Policy Optimization (GRPO)** for Medical Visual Agent.

## 🆕 RAG Integration

This model includes **Medical RAG (Retrieval-Augmented Generation)** capabilities:
- **Knowledge Base**: MedQuAD - Medical Question Answering Dataset
- **Embeddings**: sentence-transformers (all-MiniLM-L6-v2)
- **Index**: FAISS for fast similarity search

### RAG Logic:
- **Without image**: System automatically uses RAG for theoretical knowledge
- **With image**: System orchestrates RAG when lacking domain expertise

## Training Details

- **Base Model**: `{config.base_model}`
- **SFT Adapter**: `{config.sft_adapter}`
- **Method**: GRPO (DeepSeek-R1 style)
- **Group Size**: {config.group_size}
- **Steps**: {config.max_steps}
- **RAG Enabled**: {config.use_rag}

## Reward Function (with RAG Bonus)

```
R_total = 0.3×R_IoU + 0.4×R_Acc + 0.2×R_Format + R_Step + R_RAG_Bonus
```

- **R_IoU**: Detection accuracy (bounding box overlap)
- **R_Acc**: Answer accuracy
- **R_Format**: JSON structure validity
- **R_Step**: Efficiency penalty
- **R_RAG_Bonus**: +0.1 for using Medical_RAG on hard cases

## Usage

```python
from peft import PeftModel
from transformers import LlavaForConditionalGeneration

base_model = LlavaForConditionalGeneration.from_pretrained("{config.base_model}")
model = PeftModel.from_pretrained(base_model, "{REPO_ID}")
```

## Training Metrics

- Final Mean Reward: {np.mean(metrics['mean_reward'][-10:]) if metrics['mean_reward'] else 0:.4f}
- RAG Usage Rate: {np.mean(metrics['rag_usage'])*100 if metrics['rag_usage'] else 0:.1f}%
- Format Accuracy: {np.mean(metrics['format_reward'][-10:])*100 if metrics['format_reward'] else 0:.1f}%

## Citation

```
@misc{{trimedagent,
  title={{TriMedAgent: Medical Visual Agent with GRPO and RAG}},
  author={{Team}},
  year={{2025}}
}}
```
"""

with open(output_path / "README.md", 'w') as f:
    f.write(model_card)

print("✅ Model card created!")
print(f"\n🎉 GRPO Training Pipeline Complete!")
print(f"\n📋 Next: Test with notebook 03_demo.ipynb")

---
## 🎉 Done!

GRPO Training với Medical RAG hoàn tất. Model đã được push lên HuggingFace.

**Pipeline hoàn chỉnh:**
1. ✅ `01_sft_training.ipynb` - SFT với LoRA
2. ✅ `02_rl_grpo_training.ipynb` - GRPO Reinforcement Learning + Medical RAG
3. ⏭️ `03_demo.ipynb` - Demo với Gradio UI

**Tính năng mới trong version này:**
- 🧠 **Medical RAG Integration**: MedQuAD knowledge base với FAISS indexing
- 🎁 **RAG Bonus Reward**: +0.1 cho hard cases sử dụng Medical_RAG
- 📊 **Enhanced Metrics**: Track RAG usage rate và bonus rewards
- 🚀 **Demo Mode**: Test nhanh với tập nhỏ (`DEMO_MODE = True`)